In [1]:
##### Country exposure analysis: capital, labor, production in conservation priority areas, by country
# calculates the amount of each variable in priority areas (additional vs already protected) in each country
# as a % of the global priority-area total and as a % of the country's own total

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.features
import xarray as xr

In [2]:
##### SET-UP

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from helpers.reproject_raster import reproject_raster, array_to_raster

OUT_DIR = ROOT / "Results" / "30x30"
OUT_DIR.mkdir(parents=True, exist_ok=True)

##### LOAD DATA

# capital rasters
capital_path = ROOT / "Results/Raster_models/rescaled_capital_USD.tif"
capital_path_p10 = ROOT / "Results/Raster_models/rescaled_capital_USD_p10.tif"
capital_path_p90 = ROOT / "Results/Raster_models/rescaled_capital_USD_p90.tif"

# labor rasters
labor_path = ROOT / "Results/Raster_models/rescaled_jobs.tif"
labor_path_p10 = ROOT / "Results/Raster_models/rescaled_jobs_p10.tif"
labor_path_p90 = ROOT / "Results/Raster_models/rescaled_jobs_p90.tif"

# production raster
production_path = ROOT / "Data/total_production_tonnes_2020.tif"

# conservation scenario raster
# 0 = not in top 30%, 1 = in top 30%, not PA, 2 = in top 30%, already PA
biodiversity_with_PA = ROOT / "Data/conservation_scenario.tif"

# country boundaries (need to download GADM 4.1 and add to data folder https://gadm.org/data.html)
country_shp = gpd.read_file(ROOT / "Data/gadm_410-levels.gpkg", layer="ADM_0")
country_codes = pd.read_csv(ROOT / "Data/country_names.csv", encoding="cp1252")


In [3]:
##### DATA PREP

def load_summed(path):
    return array_to_raster(reproject_raster(str(path), resampling="sum"))

production = load_summed(production_path)
capital = load_summed(capital_path)
capital_p10 = load_summed(capital_path_p10)
capital_p90 = load_summed(capital_path_p90)
labor = load_summed(labor_path)
labor_p10 = load_summed(labor_path_p10)
labor_p90 = load_summed(labor_path_p90)

ci_rasters = {
    "Capital": {"p10": capital_p10, "p90": capital_p90},
    "Labor":   {"p10": labor_p10,   "p90": labor_p90},
}

value_rasters = {
    "Capital": capital,
    "Labor": labor,
    "Production": production,
}

def read_array(src, band=1):
    if isinstance(src, xr.DataArray):
        if "band" in src.dims:
            src = src.isel(band=band - 1)
        return src.squeeze().values.astype(float)

    with rasterio.open(src) as r:
        arr = r.read(band).astype(float)
        nodata = r.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
    return arr

priority_arr = read_array(biodiversity_with_PA)

In [4]:
##### RASTERIZE COUNTRIES

# merge country codes
country_shp = country_shp.merge(
    country_codes,
    left_on="GID_0",
    right_on="SHP_code",
    how="left",
)

# use the capital raster as the reference grid
ref_transform = capital.rio.transform()
ref_shape = (capital.rio.height, capital.rio.width)
ref_crs = capital.rio.crs

country_shp_proj = (
    country_shp.to_crs(ref_crs)
    if country_shp.crs != ref_crs
    else country_shp.copy()
).reset_index(drop=True)

shapes = (
    (geom, i)
    for i, geom in enumerate(country_shp_proj.geometry)
    if geom is not None
)

country_id_arr = rasterio.features.rasterize(
    shapes=shapes,
    out_shape=ref_shape,
    transform=ref_transform,
    fill=-1,
    dtype="int32",
)

n_countries = len(country_shp_proj)
row_to_iso3 = country_shp_proj["ISO3"].to_dict()

# country name lookup (carried through to outputs so figure scripts don't need the shapefile)
name_col = next(
    (c for c in ["COUNTRY", "NAME_0", "Country", "country_name"] if c in country_shp_proj.columns),
    None,
)
if name_col:
    name_lookup = (
        country_shp_proj[["ISO3", name_col]]
        .drop_duplicates("ISO3")
        .set_index("ISO3")[name_col]
    )
else:
    name_lookup = pd.Series(dtype=object)

In [5]:
##### ZONAL SUMS

def zonal_sum_by_country(value_arr, priority_arr, class_value, country_id_arr, n_countries):
    valid = (
        (country_id_arr >= 0)
        & (priority_arr == class_value)
        & ~np.isnan(value_arr)
    )
    return np.bincount(
        country_id_arr[valid],
        weights=value_arr[valid],
        minlength=n_countries,
    )

# national totals
national_totals = {}

for variable, raster in value_rasters.items():
    arr = read_array(raster)
    valid = (country_id_arr >= 0) & ~np.isnan(arr)
    totals = np.bincount(
        country_id_arr[valid],
        weights=arr[valid],
        minlength=n_countries,
    )
    national_totals[variable] = {row_to_iso3[i]: totals[i] for i in range(n_countries)}

# country-level priority totals
country_rows = []

for variable, raster in value_rasters.items():
    arr = read_array(raster)

    for class_value, class_name in [(1, "Additional priority"), (2, "Already in PA")]:
        sums = zonal_sum_by_country(arr, priority_arr, class_value, country_id_arr, n_countries)

        for i in range(n_countries):
            iso3 = row_to_iso3[i]
            country_rows.append({
                "ISO3": iso3,
                "Variable": variable,
                "Class": class_name,
                "Sum": sums[i],
                "National_Total": national_totals[variable].get(iso3, np.nan),
            })

country_df = pd.DataFrame(country_rows)

# priority-area-only global totals
in_priority = (priority_arr == 1) | (priority_arr == 2)
global_priority_totals = {
    variable: np.nansum(read_array(raster)[in_priority])
    for variable, raster in value_rasters.items()
}

country_df["Pct_of_national"] = np.where(
    country_df["National_Total"] > 0,
    100 * country_df["Sum"] / country_df["National_Total"],
    np.nan,
)
country_df["Global_Total"] = country_df["Variable"].map(global_priority_totals)
country_df["Pct_of_global"] = 100 * country_df["Sum"] / country_df["Global_Total"]
country_df.insert(1, "Country", country_df["ISO3"].map(name_lookup))

In [6]:
##### UNCERTAINTY (p10/p90)

ci_rows = []

for variable, quantiles in ci_rasters.items():
    for q, raster in quantiles.items():
        arr = read_array(raster)
        sums = (
            zonal_sum_by_country(arr, priority_arr, 1, country_id_arr, n_countries)
            + zonal_sum_by_country(arr, priority_arr, 2, country_id_arr, n_countries)
        )
        for i in range(n_countries):
            iso3 = row_to_iso3[i]
            nat_total = national_totals[variable].get(iso3, np.nan)
            ci_rows.append({
                "ISO3": iso3,
                "Variable": variable,
                "Quantile": q,
                "Pct_of_global": 100 * sums[i] / global_priority_totals[variable],
                "Pct_of_national": 100 * sums[i] / nat_total if nat_total > 0 else np.nan,
            })

ci_df = pd.DataFrame(ci_rows)

In [7]:
##### SUMMARY TABLE

country_totals = (
    country_df
    .groupby(["ISO3", "Variable"], as_index=False)
    .agg(
        Priority_Total=("Sum", "sum"),
        National_Total=("National_Total", "first"),
        Pct_of_national=("Pct_of_national", "sum"),
        Pct_of_global=("Pct_of_global", "sum"),
    )
)

export_df = country_totals.pivot(
    index="ISO3",
    columns="Variable",
    values=["Pct_of_global", "Pct_of_national"],
)
export_df.columns = [f"{metric}_{variable}" for metric, variable in export_df.columns]
export_df = export_df.reset_index()

if name_col:
    export_df.insert(1, "Country", export_df["ISO3"].map(name_lookup))

export_df = export_df.rename(columns={
    "Pct_of_global_Capital": "Pct_global_capital_in_country",
    "Pct_of_global_Labor": "Pct_global_labor_in_country",
    "Pct_of_global_Production": "Pct_global_production_in_country",
    "Pct_of_national_Capital": "Pct_national_capital_exposed",
    "Pct_of_national_Labor": "Pct_national_labor_exposed",
    "Pct_of_national_Production": "Pct_national_production_exposed",
})

ordered_cols = ["ISO3", "Country"] if name_col else ["ISO3"]
ordered_cols += [
    "Pct_global_capital_in_country",
    "Pct_global_labor_in_country",
    "Pct_global_production_in_country",
    "Pct_national_capital_exposed",
    "Pct_national_labor_exposed",
    "Pct_national_production_exposed",
]
ordered_cols = [c for c in ordered_cols if c in export_df.columns]
export_df = export_df[ordered_cols]

export_df = export_df[export_df["ISO3"] != "OTHER"].sort_values(
    "Pct_global_capital_in_country", ascending=False
)

##### SAVE
country_df.to_csv(OUT_DIR / "country_exposure_long.csv", index=False)
ci_df.to_csv(OUT_DIR / "country_exposure_ci.csv", index=False)
export_df.to_csv(OUT_DIR / "country_exposure_summary.csv", index=False)